# Healthcare Analytics Demo — One-Click Launcher

**Run All** to deploy the complete Healthcare Payer/Provider analytics solution into this Fabric workspace.

### What Gets Deployed
| Layer | Items |
|-------|-------|
| **Lakehouses** | `lh_bronze_raw`, `lh_silver_stage`, `lh_silver_ods`, `lh_gold_curated` |
| **Notebooks** | 5 ETL + 2 data generation + 5 RTI (event simulator, Eventhouse setup, 3 scoring) |
| **Pipelines** | `PL_Healthcare_Full_Load`, `PL_Healthcare_Master` |
| **Semantic Model** | `HealthcareDemoHLS` (star schema) |
| **Data Agent** | `HealthcareHLSAgent` (Copilot AI) |
| **Ontology + Graph** | `Healthcare_Demo_Ontology_HLS` + `Healthcare_Demo_Graph` (deployed via API after pipeline) |
| **Real-Time Intelligence** | Eventhouse + KQL DB + 3 scoring use cases (fraud, care gaps, high-cost trajectory) |

### Prerequisites
- An **empty** Fabric workspace (this notebook should be the only item)
- Fabric capacity: **F64** or higher recommended
- User must be workspace **Admin** or **Member**

### Configuration
Edit the cell below to point to your GitHub repo (public or private).

In [ ]:
# ============================================================================
# CONFIGURATION — Edit these values
# ============================================================================

GITHUB_OWNER = "rasgiza"                # GitHub org or user (public mirror)
GITHUB_REPO  = "Fabric-Payer-Provider-HealthCare-Demo"
GITHUB_BRANCH = "main"
GITHUB_TOKEN = ""                    # Leave empty for public repos. Only needed for private repos (classic PAT with repo scope).

# Deployment options
GENERATE_DATA = True                  # Generate fresh synthetic data
RUN_PIPELINE = True                   # Run the full-load pipeline after data gen
UPLOAD_KNOWLEDGE_DOCS = True          # Upload healthcare knowledge docs to lakehouse
DEPLOY_RTI = True                     # Deploy Real-Time Intelligence (Eventhouse + scoring)

print(f"Will deploy from: github.com/{GITHUB_OWNER}/{GITHUB_REPO} @ {GITHUB_BRANCH}")

In [ ]:
# ============================================================================
# CELL 1 — Install fabric-launcher
# ============================================================================

%pip install fabric-launcher --quiet

import notebookutils
from fabric_launcher import FabricLauncher

print("fabric-launcher installed successfully")

In [ ]:
# ============================================================================
# CELL 2 — Initialize launcher
# ============================================================================

launcher = FabricLauncher(
    notebookutils,
    environment="DEV",
    allow_non_empty_workspace=False,
    fix_zero_logical_ids=True,
    debug=False,
)

workspace_id = notebookutils.runtime.context.get("currentWorkspaceId", "")
print(f"Workspace ID: {workspace_id}")
print(f"Launcher ready")

In [ ]:
# ============================================================================
# CELL 3 — Download repo & deploy all Fabric artifacts from scratch (staged)
# ============================================================================
# Downloads the repo as a ZIP, extracts it, and creates every artifact
# in the workspace using the Fabric REST API.  Order matters because
# later items reference earlier ones via logicalId.
#
# Stage 1: Lakehouses          — must exist before notebooks reference them
# Stage 2: Eventhouse           — async provisioning, needs its own stage
# Stage 3: KQL Database         — depends on Eventhouse being fully ready
# Stage 4: Notebooks (create)  — creates notebook items in workspace
# Stage 5: Notebooks (content) — updateDefinition applies code reliably
#           (Fabric Create Item API can silently drop inline definitions
#            for notebooks; a second pass forces updateDefinition which
#            always applies the content.)
# Stage 6: Data Pipelines      — orchestrate notebook execution
# Stage 7: Semantic Model + Data Agent — need Gold tables populated first
# ============================================================================

print("Downloading repo and deploying artifacts...")
print("This takes 3-5 minutes.\n")

downloader, deployer, report = launcher.download_and_deploy(
    repo_owner=GITHUB_OWNER,
    repo_name=GITHUB_REPO,
    branch=GITHUB_BRANCH,
    github_token=GITHUB_TOKEN if GITHUB_TOKEN else None,
    workspace_folder="workspace",
    item_type_stages=[
        ["Lakehouse"],                         # Stage 1
        ["Eventhouse"],                        # Stage 2 — provision Eventhouse first
        ["KQLDatabase"],                       # Stage 3 — KQL DB needs Eventhouse ready
        ["Notebook"],                          # Stage 4 — create notebook items
        ["Notebook"],                          # Stage 5 — updateDefinition ensures code is applied
        ["DataPipeline"],                      # Stage 6
        ["SemanticModel", "DataAgent"],        # Stage 7
    ],
    validate_after_deployment=True,
    generate_report=True,
    deployment_retries=2,
)

print("\n" + "=" * 60)
print("  ARTIFACT DEPLOYMENT COMPLETE")
print("=" * 60)

In [ ]:
# ============================================================================
# CELL 3b — Push notebook content via ipynb format (guaranteed to work)
# ============================================================================
# The Fabric Create Item API silently creates empty notebook shells.
# The generic updateDefinition with .py format returns 200 but doesn't apply.
# Solution: Convert .py → .ipynb format and push via the notebook-specific
# updateDefinition API, which documents ipynb as the supported format.
# ============================================================================

import base64, requests, os, json, time, re

print("Converting notebooks to ipynb and pushing via REST API...\n")

token = notebookutils.credentials.getToken("pbi")
hdrs = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
api = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

# ── Helper: Convert Fabric .py source → ipynb JSON ──────────────────────
def py_to_ipynb(py_content, metadata_json=None, logical_to_real=None, ws_id=None):
    """Convert Fabric notebook .py source to Jupyter ipynb JSON."""
    # Replace %pip magic with subprocess (disabled in child notebooks via notebookutils.notebook.run)
    py_content = re.sub(
        r'^%pip install (.+)$',
        r'import subprocess, sys; subprocess.check_call([sys.executable, "-m", "pip", "install"] + "\1".split())',
        py_content,
        flags=re.MULTILINE,
    )
    lines = py_content.split("\n")
    cells = []
    i = 0

    # Skip header
    while i < len(lines) and (lines[i].strip() in ("", "# Fabric notebook source")):
        i += 1

    while i < len(lines):
        # Look for METADATA marker
        if lines[i].startswith("# METADATA **"):
            i += 1
            while i < len(lines) and lines[i].strip() == "":
                i += 1
            if i >= len(lines):
                break

            # Determine cell type
            if lines[i].startswith("# CELL **"):
                cell_type = "code"
            elif lines[i].startswith("# MARKDOWN **"):
                cell_type = "markdown"
            else:
                i += 1
                continue

            i += 1
            # Skip blank line after marker
            if i < len(lines) and lines[i].strip() == "":
                i += 1

            # Collect content until next METADATA or EOF
            cell_lines = []
            while i < len(lines) and not lines[i].startswith("# METADATA **"):
                cell_lines.append(lines[i])
                i += 1

            # Remove trailing blank lines
            while cell_lines and cell_lines[-1].strip() == "":
                cell_lines.pop()

            # For markdown: remove "# " prefix
            if cell_type == "markdown":
                processed = []
                for ln in cell_lines:
                    if ln.startswith("# "):
                        processed.append(ln[2:])
                    elif ln == "#":
                        processed.append("")
                    else:
                        processed.append(ln)
                cell_lines = processed

            if cell_lines:
                # Build source array (each line gets \n except the last)
                source = [ln + "\n" for ln in cell_lines[:-1]]
                source.append(cell_lines[-1])

                nb_cell = {"cell_type": cell_type, "metadata": {}, "source": source}
                if cell_type == "code":
                    nb_cell["outputs"] = []
                    nb_cell["execution_count"] = None
                cells.append(nb_cell)
        else:
            i += 1

    # Build notebook structure
    nb_metadata = {
        "kernel_info": {"name": "synapse_pyspark"},
        "kernelspec": {
            "name": "synapse_pyspark",
            "display_name": "Synapse PySpark",
            "language": "Python",
        },
        "language_info": {"name": "python"},
    }

    # Merge lakehouse dependencies from metadata JSON
    if metadata_json:
        deps = metadata_json.get("dependencies", {})
        if deps:
            # Replace logical IDs in the metadata
            deps_str = json.dumps(deps)
            if logical_to_real:
                for lid, rid in logical_to_real.items():
                    deps_str = deps_str.replace(lid, rid)
            if ws_id:
                deps_str = deps_str.replace("00000000-0000-0000-0000-000000000000", ws_id)
            nb_metadata["dependencies"] = json.loads(deps_str)

    return json.dumps(
        {"nbformat": 4, "nbformat_minor": 5, "metadata": nb_metadata, "cells": cells},
        indent=1,
    )

# ── Step 1: Get all deployed items ──────────────────────────────────────
resp = requests.get(f"{api}/items", headers=hdrs)
all_items = resp.json().get("value", [])
name_to_id = {(it["type"], it["displayName"]): it["id"] for it in all_items}

# ── Step 2: Locate extracted workspace directory ────────────────────────
ws_dir = None
for candidate in [".lakehouse/default/Files/src", "/lakehouse/default/Files/src"]:
    d = os.path.join(candidate, "workspace")
    if os.path.isdir(d):
        ws_dir = d
        break

if not ws_dir:
    print("ERROR: Could not find extracted workspace directory.")
else:
    # Build logical_id → real_id mapping
    logical_to_real = {}
    for entry in os.listdir(ws_dir):
        pf = os.path.join(ws_dir, entry, ".platform")
        if os.path.exists(pf):
            with open(pf, "r") as f:
                plat = json.load(f)
            itype = plat["metadata"]["type"]
            iname = plat["metadata"]["displayName"]
            lid = plat["config"]["logicalId"]
            rid = name_to_id.get((itype, iname), "")
            if lid and rid:
                logical_to_real[lid] = rid

    # ── Step 3: Convert and push each notebook ──────────────────────────
    pushed, failed = 0, 0
    for entry in sorted(os.listdir(ws_dir)):
        if not entry.endswith(".Notebook"):
            continue
        nb_name = entry.replace(".Notebook", "")
        nb_id = name_to_id.get(("Notebook", nb_name))
        if not nb_id:
            print(f"  SKIP {nb_name}: not in workspace")
            continue

        nb_path = os.path.join(ws_dir, entry)

        # Read .py content
        py_file = os.path.join(nb_path, "notebook-content.py")
        if not os.path.exists(py_file):
            print(f"  SKIP {nb_name}: no notebook-content.py")
            continue
        with open(py_file, "r", encoding="utf-8") as f:
            py_content = f.read()

        # Read metadata JSON
        meta_file = os.path.join(nb_path, ".metadata", "notebook-metadata.json")
        meta_json = None
        if os.path.exists(meta_file):
            with open(meta_file, "r", encoding="utf-8") as f:
                meta_json = json.load(f)

        # Convert to ipynb
        ipynb_content = py_to_ipynb(py_content, meta_json, logical_to_real, workspace_id)
        ipynb_b64 = base64.b64encode(ipynb_content.encode("utf-8")).decode("utf-8")

        # Build payload — single part: the ipynb file
        parts = [
            {"path": "notebook-content.ipynb", "payload": ipynb_b64, "payloadType": "InlineBase64"}
        ]
        body = {"definition": {"format": "ipynb", "parts": parts}}

        # POST to updateDefinition (with retry for transient 500s)
        r = None
        for attempt in range(3):
            r = requests.post(
                f"{api}/items/{nb_id}/updateDefinition",
                headers=hdrs,
                json=body,
            )
            if r.status_code in (200, 202):
                break
            if r.status_code >= 500 and attempt < 2:
                print(f"  {nb_name}: HTTP {r.status_code}, retrying ({attempt+1}/3)...")
                time.sleep(5 * (attempt + 1))
            else:
                break

        # Handle LRO (202)
        if r.status_code == 202:
            loc = r.headers.get("Location", "")
            retry_after = int(r.headers.get("Retry-After", "3"))
            if loc:
                for _ in range(60):
                    time.sleep(retry_after)
                    pr = requests.get(loc, headers=hdrs)
                    if pr.status_code == 200:
                        break
                    elif pr.status_code != 202:
                        break

        if r.status_code in (200, 202):
            # Count cells for feedback
            cell_count = ipynb_content.count('"cell_type"')
            size_kb = round(len(ipynb_content) / 1024, 1)
            pushed += 1
            print(f"  {nb_name}: OK ({cell_count} cells, {size_kb}KB)")
        else:
            failed += 1
            print(f"  FAIL {nb_name}: HTTP {r.status_code} — {r.text[:300]}")

    print(f"\n{'='*60}")
    print(f"  Notebooks pushed: {pushed} (ipynb format)")
    if failed:
        print(f"  Failed: {failed}")

    # ── Step 4: Verify ──────────────────────────────────────────────────
    print(f"\nVerifying content (getDefinition on 01_Bronze_Ingest_CSV)...")
    test_id = name_to_id.get(("Notebook", "01_Bronze_Ingest_CSV"))
    if test_id:
        vr = requests.post(f"{api}/items/{test_id}/getDefinition", headers=hdrs, json={})
        # Handle LRO for getDefinition
        if vr.status_code == 202:
            loc = vr.headers.get("Location", "")
            if loc:
                for _ in range(30):
                    time.sleep(2)
                    vr = requests.get(loc, headers=hdrs)
                    if vr.status_code == 200:
                        break

        if vr.status_code == 200:
            vdata = vr.json()
            defn_parts = vdata.get("definition", {}).get("parts", [])
            print(f"  Parts returned: {[p.get('path') for p in defn_parts]}")
            for p in defn_parts:
                if "content" in p.get("path", "").lower():
                    decoded = base64.b64decode(p["payload"]).decode("utf-8")
                    has_spark = "spark" in decoded.lower() or "bronze" in decoded.lower()
                    size = len(decoded)
                    print(f"  Content size: {size:,} bytes, has_code: {has_spark}")
                    if size > 500:
                        print(f"  ✅ Notebook content verified — has real code")
                    else:
                        print(f"  ⚠️ Content is suspiciously small: {decoded[:200]}")
                    break
            else:
                print(f"  ⚠️ No content part found. All parts: {json.dumps([{k:v for k,v in p.items() if k != 'payload'} for p in defn_parts], indent=2)}")
        else:
            print(f"  getDefinition returned HTTP {vr.status_code}: {vr.text[:200]}")
    print(f"{'='*60}")

In [ ]:
# ============================================================================
# CELL 4 — Upload healthcare knowledge docs to Data Agent lakehouse
# ============================================================================

if UPLOAD_KNOWLEDGE_DOCS:
    import os, glob

    # fabric-launcher extracts the repo ZIP to this relative path by default
    # Try both relative (library default) and absolute (Fabric mount) paths
    for candidate in [".lakehouse/default/Files/src", "/lakehouse/default/Files/src"]:
        if os.path.isdir(candidate):
            extract_base = candidate
            break
    else:
        print("Warning: Could not find extract directory. Skipping knowledge doc upload.")
        print("  Tried: .lakehouse/default/Files/src and /lakehouse/default/Files/src")
        UPLOAD_KNOWLEDGE_DOCS = False
        extract_base = None

    # fabric-launcher extracts repo contents directly into extract_base (no wrapper folder)
    repo_root = extract_base

    knowledge_src = os.path.join(repo_root, "healthcare_knowledge") if repo_root else None

    if knowledge_src and os.path.exists(knowledge_src):
        print("Uploading healthcare knowledge documents...")
        launcher.upload_files_to_lakehouse(
            lakehouse_name="lh_gold_curated",
            source_directory=knowledge_src,
            target_folder="healthcare_knowledge",
            file_patterns=["*.md"],
        )
        # Count uploaded files
        count = sum(len(files) for _, _, files in os.walk(knowledge_src) if files)
        print(f"Uploaded {count} knowledge documents to lh_gold_curated/Files/healthcare_knowledge/")
    else:
        print(f"Warning: healthcare_knowledge folder not found at {knowledge_src}")
else:
    print("Skipping knowledge doc upload (UPLOAD_KNOWLEDGE_DOCS=False)")

In [ ]:
# ============================================================================
# CELL 5 — Generate synthetic data
# ============================================================================
# Uses notebookutils.notebook.run() — the native Fabric way to orchestrate
# notebooks. More reliable than the Jobs REST API after updateDefinition.
# ============================================================================

if GENERATE_DATA:
    print("Running NB_Generate_Sample_Data (generates ~10K patients, 100K encounters)...")
    print("This takes 2-4 minutes.\n")

    try:
        notebookutils.notebook.run("NB_Generate_Sample_Data", 1200, {"useRootDefaultLakehouse": True})
        print("\n✅ Data generation SUCCEEDED")
    except Exception as e:
        print(f"\n❌ Data generation FAILED: {e}")
        print("Try running NB_Generate_Sample_Data manually from the workspace.")
else:
    print("Skipping data generation (GENERATE_DATA=False)")

In [ ]:
# ============================================================================
# CELL 6 — Run the full-load pipeline
# ============================================================================

if RUN_PIPELINE:
    import requests, time, json

    token = notebookutils.credentials.getToken("pbi")
    headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
    api_base = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

    # Find the pipeline ID
    print("Looking up PL_Healthcare_Master pipeline...")
    resp = requests.get(f"{api_base}/items?type=DataPipeline", headers=headers)
    resp.raise_for_status()
    pipelines = resp.json().get("value", [])
    pipeline = next((p for p in pipelines if p["displayName"] == "PL_Healthcare_Master"), None)

    if not pipeline:
        print("WARNING: PL_Healthcare_Master not found. Skipping pipeline run.")
        print("You can run it manually from the workspace.")
    else:
        pipeline_id = pipeline["id"]
        print(f"Pipeline ID: {pipeline_id}")

        # Trigger pipeline with load_mode=full
        print("Triggering pipeline with load_mode=full...")
        trigger_body = {
            "executionData": {
                "parameters": {
                    "load_mode": "full"
                }
            }
        }
        resp = requests.post(
            f"{api_base}/items/{pipeline_id}/jobs/instances?jobType=Pipeline",
            headers=headers,
            json=trigger_body,
        )

        if resp.status_code in (200, 202):
            # Get job ID from Location header or response
            location = resp.headers.get("Location", "")
            print(f"Pipeline triggered. Polling for completion...")
            print(f"(This takes 8-15 minutes for full load)\n")

            # Poll until complete
            max_polls = 120  # 120 * 15s = 30 min max
            for poll in range(max_polls):
                time.sleep(15)
                try:
                    if location:
                        poll_resp = requests.get(location, headers=headers)
                    else:
                        poll_resp = requests.get(
                            f"{api_base}/items/{pipeline_id}/jobs/instances",
                            headers=headers,
                        )
                    if poll_resp.status_code == 200:
                        job_data = poll_resp.json()
                        status = job_data.get("status", "Unknown")
                        if status in ("Completed", "Succeeded"):
                            print(f"  Pipeline COMPLETED after {(poll+1)*15}s")
                            break
                        elif status in ("Failed", "Cancelled"):
                            print(f"  Pipeline {status} after {(poll+1)*15}s")
                            print(f"  Check the pipeline run history for details.")
                            break
                        else:
                            if poll % 4 == 0:  # Print every 60s
                                print(f"  [{(poll+1)*15}s] Status: {status}...")
                except Exception as e:
                    if poll % 4 == 0:
                        print(f"  [{(poll+1)*15}s] Polling... ({e})")
            else:
                print("  Pipeline still running after 30 min. Check workspace for status.")
        else:
            print(f"  Pipeline trigger returned {resp.status_code}: {resp.text}")
            print("  You can run it manually from the workspace.")
else:
    print("Skipping pipeline run (RUN_PIPELINE=False)")
    print("To run manually: Open PL_Healthcare_Master → Run with load_mode=full")

In [ ]:
# ============================================================================
# CELL 6b — Refresh Semantic Model (Direct Lake)
# ============================================================================
# The semantic model (HealthcareDemoHLS) was deployed as a Git artifact in
# Cell 3, but Direct Lake needs Gold tables populated first. Now that the
# pipeline has finished, trigger a refresh so the model binds to the tables.
# This MUST happen BEFORE ontology deployment (Cell 7) since the ontology
# references the semantic model's tables.
# ============================================================================

import requests, time

token = notebookutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
api_base = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

print("=" * 60)
print("  SEMANTIC MODEL REFRESH")
print("=" * 60)

# Find the semantic model
resp = requests.get(f"{api_base}/items?type=SemanticModel", headers=headers)
resp.raise_for_status()
models = resp.json().get("value", [])
sm = next((m for m in models if m["displayName"] == "HealthcareDemoHLS"), None)

if not sm:
    print("  [WARN] HealthcareDemoHLS not found — skipping refresh")
    print("  You can refresh it manually from the workspace.")
else:
    sm_id = sm["id"]
    print(f"  Found: HealthcareDemoHLS ({sm_id})")
    print(f"  Triggering refresh...")

    # Use Power BI enhanced refresh API (Fabric /semanticModels endpoint returns 404)
    pbi_base = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}"
    refresh_url = f"{pbi_base}/datasets/{sm_id}/refreshes"
    refresh_body = {"type": "Full"}
    r = requests.post(refresh_url, headers=headers, json=refresh_body)

    if r.status_code in (200, 202):
        print(f"  Refresh triggered. Polling for completion...")
        # Poll for completion
        for poll in range(60):  # 60 * 10s = 10 min max
            time.sleep(10)
            poll_resp = requests.get(refresh_url, headers=headers)
            if poll_resp.status_code == 200:
                refreshes = poll_resp.json().get("value", [])
                if refreshes:
                    latest = refreshes[0]
                    status = latest.get("status", "Unknown")
                    if status in ("Completed", "Succeeded"):
                        print(f"  Refresh COMPLETED after {(poll+1)*10}s")
                        break
                    elif status == "Failed":
                        print(f"  Refresh FAILED after {(poll+1)*10}s")
                        err = latest.get("extendedStatus", "")
                        if err:
                            print(f"  Error: {err}")
                        break
                    elif poll % 3 == 0:
                        print(f"  [{(poll+1)*10}s] Status: {status}...")
        else:
            print("  Refresh still running after 10 min. Check workspace for status.")
    else:
        print(f"  Refresh trigger returned HTTP {r.status_code}: {r.text[:300]}")
        print("  You can refresh the model manually from the workspace.")

print("=" * 60)

In [ ]:
# ============================================================================
# CELL 7 — Deploy Ontology & Graph Model via API
# ============================================================================
# Deploys the ontology (10 entity types, 14 relationships) and a standalone
# graph model from the ontology metadata on disk.
#
# The Fabric REST API does NOT auto-provision a graph model underneath the
# ontology (unlike the Fabric UI), so this cell deploys both:
#   1. Ontology — entity types + relationships + data bindings to lh_gold_curated
#   2. Graph Model — 5-part definition via 2-step create pattern
# ============================================================================

import subprocess, sys, os

print("=" * 60)
print("  ONTOLOGY & GRAPH MODEL — API Deployment")
print("=" * 60)
print()

# Resolve paths — the repo was extracted by fabric-launcher
for candidate in [".lakehouse/default/Files/src", "/lakehouse/default/Files/src"]:
    if os.path.isdir(candidate):
        repo_root = candidate
        break
else:
    repo_root = None
    print("  [WARN] Could not find extracted repo. Skipping ontology deployment.")
    print("  You can deploy manually:")
    print("    python scripts/deploy_ontology.py --workspace <name> --tenant-id <id>")

if repo_root:
    deploy_script = os.path.join(repo_root, "scripts", "deploy_ontology.py")
    if os.path.exists(deploy_script):
        # Get workspace name from the Fabric context
        try:
            ws_name = notebookutils.runtime.context.get("currentWorkspaceName", "")
        except Exception:
            ws_name = ""

        if not ws_name:
            print("  [WARN] Could not resolve workspace name from Fabric context.")
            print("  Set ws_name manually and re-run this cell.")
        else:
            # Get tenant ID from token claims
            import json as _json, base64 as _b64
            token = notebookutils.credentials.getToken("pbi")
            try:
                payload = token.split(".")[1]
                payload += "=" * (4 - len(payload) % 4)  # pad base64
                claims = _json.loads(_b64.b64decode(payload))
                tenant_id = claims.get("tid", "")
            except Exception:
                tenant_id = ""

            if not tenant_id:
                print("  [WARN] Could not extract tenant ID from token.")
                print("  Set tenant_id manually and re-run this cell.")
            else:
                print(f"  Workspace: {ws_name}")
                print(f"  Tenant:    {tenant_id}")
                print()

                # Run deploy_ontology.py (which auto-chains graph model deployment)
                cmd = [
                    sys.executable, deploy_script,
                    "--workspace", ws_name,
                    "--tenant-id", tenant_id,
                ]
                print(f"  Running: {' '.join(cmd)}")
                print()
                result = subprocess.run(cmd, capture_output=False, text=True)

                if result.returncode == 0:
                    print()
                    print("  \u2705 Ontology + Graph Model deployed successfully")
                else:
                    print()
                    print(f"  \u274c Deployment failed (exit code {result.returncode})")
                    print("  Check the output above for details.")
    else:
        print(f"  [WARN] deploy_ontology.py not found at {deploy_script}")
        print("  Ensure the repo was extracted correctly.")

print()
print("=" * 60)


In [ ]:
# ============================================================================
# CELL 7b — Deploy Real-Time Intelligence (RTI)
# ============================================================================
# Eventhouse + KQL Database are already deployed as Git artifacts (Stage 2).
# This cell runs post-deploy wiring and scoring notebooks:
#   1. NB_RTI_Setup_Eventhouse  — Discovers artifacts, creates tables, creates Eventstream
#   2. NB_RTI_Event_Simulator   — Generates initial events (batch mode)
#   3. NB_RTI_Fraud_Detection   — Scores claims for fraud patterns
#   4. NB_RTI_Care_Gap_Alerts   — Point-of-care gap closure alerts
#   5. NB_RTI_HighCost_Trajectory — High-cost member trajectory analysis
# ============================================================================
# Uses notebookutils.notebook.run() for reliable orchestration.
# ============================================================================

if DEPLOY_RTI:
    print("=" * 60)
    print("  REAL-TIME INTELLIGENCE DEPLOYMENT")
    print("=" * 60)

    rti_notebooks = [
        "NB_RTI_Setup_Eventhouse",
        "NB_RTI_Event_Simulator",
        "NB_RTI_Fraud_Detection",
        "NB_RTI_Care_Gap_Alerts",
        "NB_RTI_HighCost_Trajectory",
    ]

    for nb_name in rti_notebooks:
        print(f"\n  Running {nb_name}...")
        try:
            notebookutils.notebook.run(nb_name, 1200, {"useRootDefaultLakehouse": True})
            print(f"  -> {nb_name}: ✅ OK")
        except Exception as e:
            print(f"  -> {nb_name}: ❌ FAILED -- {e}")
            print(f"    You can run this notebook manually from the workspace.")

    print("\n" + "=" * 60)
    print("  RTI DEPLOYMENT COMPLETE")
    print("=" * 60)
    print("  Tables created in lh_gold_curated:")
    print("    - rti_claims_events, rti_adt_events, rti_rx_events")
    print("    - rti_fraud_scores, rti_care_gap_alerts, rti_highcost_alerts")
    print()
    print("  Eventhouse: Healthcare_RTI_Eventhouse (+ KQL DB)")
    print("  For streaming mode: set MODE='stream' in NB_RTI_Event_Simulator")
    print("=" * 60)
else:
    print("Skipping RTI deployment (DEPLOY_RTI=False)")

In [ ]:
# ============================================================================
# CELL 8 — Deployment Summary
# ============================================================================

import requests

token = notebookutils.credentials.getToken("pbi")
headers = {"Authorization": f"Bearer {token}"}
api_base = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

resp = requests.get(f"{api_base}/items", headers=headers)
items = resp.json().get("value", []) if resp.status_code == 200 else []

# Group by type
by_type = {}
for item in items:
    t = item.get("type", "Unknown")
    by_type.setdefault(t, []).append(item["displayName"])

print("=" * 60)
print("  DEPLOYMENT COMPLETE")
print("=" * 60)
print(f"  Workspace: {workspace_id}")
print(f"  Total items: {len(items)}")
print()
for item_type in ["Lakehouse", "Notebook", "DataPipeline", "SemanticModel", "DataAgent", "Ontology", "Eventhouse", "KQLDatabase", "Eventstream", "SQLEndpoint"]:
    names = by_type.get(item_type, [])
    if names:
        print(f"  {item_type} ({len(names)}):")
        for n in sorted(names):
            print(f"    - {n}")

# Show any other types
shown = {"Lakehouse", "Notebook", "DataPipeline", "SemanticModel", "DataAgent", "Ontology", "Eventhouse", "KQLDatabase", "Eventstream", "SQLEndpoint"}
for item_type, names in by_type.items():
    if item_type not in shown:
        print(f"  {item_type} ({len(names)}):")
        for n in sorted(names):
            print(f"    - {n}")

# RTI table counts
print()
print("  RTI Delta Tables (lh_gold_curated):")
try:
    for tbl in ["rti_claims_events", "rti_adt_events", "rti_rx_events", "rti_fraud_scores", "rti_care_gap_alerts", "rti_highcost_alerts"]:
        try:
            cnt = spark.table(tbl).count()
            print(f"    - {tbl}: {cnt:,} rows")
        except Exception:
            print(f"    - {tbl}: (not created yet)")
except Exception:
    print("    (could not query RTI tables)")

print()
print("=" * 60)
print("  NEXT STEPS")
print("=" * 60)
print("  1. Open the HealthcareDemoHLS semantic model -> verify tables loaded")
print("  2. Open the HealthcareHLSAgent -> ask: 'What are the top denial reasons?'")
print("  3. Verify the Ontology + Graph Model (deployed after pipeline)")
print("     -> Open Healthcare_Demo_Ontology_HLS in workspace to explore")
print("  4. For incremental loads: run PL_Healthcare_Master with load_mode=incremental")
print("     (auto-generates incremental data + runs full ETL in one shot)")
print("  5. RTI: Open Healthcare_RTI_Eventhouse to explore real-time data")
print("     - NB_RTI_Fraud_Detection: View rti_fraud_scores for flagged claims")
print("     - NB_RTI_Care_Gap_Alerts: View rti_care_gap_alerts for gap closures")
print("     - NB_RTI_HighCost_Trajectory: View rti_highcost_alerts for cost trends")
print("  6. For live streaming: set MODE='stream' in NB_RTI_Event_Simulator")
print("     and configure the Eventstream Custom App connection string")
print("=" * 60)